# NB1A - Pokemon Data Collection
This notebook will include the code utilised to collect the data from the PokeAPI. This is the link to the PokeAPI [documentation](https://pokeapi.co/docs/v2). It is of note, however, that some of the doucmentation is outdated and not accurate. These will be highlighted when encountered. 

In [33]:
import json
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

In [34]:
def get_generations_pokemon(generation: int):
    request_url = f'https://pokeapi.co/api/v2/generation/{generation}'
    response = requests.get(request_url)
    data = response.json()
    pokemon_df = pd.json_normalize(data, record_path = 'pokemon_species')
    pokemon_df['generation'] = generation
    return pokemon_df

In [35]:
list_of_generations = [1, 2, 3, 4, 5, 6, 7, 8, 9]
list_of_df = [get_generations_pokemon(generation) for generation in list_of_generations]
poke_df = pd.concat(list_of_df)
display(poke_df)


,name,url,generation
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/,1
1,charmander,https://pokeapi.co/api/v2/pokemon-species/4/,1
2,squirtle,https://pokeapi.co/api/v2/pokemon-species/7/,1
3,caterpie,https://pokeapi.co/api/v2/pokemon-species/10/,1
4,weedle,https://pokeapi.co/api/v2/pokemon-species/13/,1
...,...,...,...
115,gholdengo,https://pokeapi.co/api/v2/pokemon-species/1000/,9
116,dipplin,https://pokeapi.co/api/v2/pokemon-species/1011/,9
117,sinistcha,https://pokeapi.co/api/v2/pokemon-species/1013/,9
118,archaludon,https://pokeapi.co/api/v2/pokemon-species/1018/,9


In [36]:
def extract_pokemon_id(url): 
    id = url.rstrip('/').split('/')[-1]
    return int(id)

In [37]:
poke_df['pokemon_id'] = poke_df['url'].apply(extract_pokemon_id)
poke_df = poke_df.sort_values('pokemon_id').reset_index(drop = True)
display(poke_df)
poke_df.to_json('../../data/Pokemon Data/generation_pokemon/generation_pokemon.json')


,name,url,generation,pokemon_id
0,bulbasaur,https://pokeapi.co/api/v2/pokemon-species/1/,1,1
1,ivysaur,https://pokeapi.co/api/v2/pokemon-species/2/,1,2
2,venusaur,https://pokeapi.co/api/v2/pokemon-species/3/,1,3
3,charmander,https://pokeapi.co/api/v2/pokemon-species/4/,1,4
4,charmeleon,https://pokeapi.co/api/v2/pokemon-species/5/,1,5
...,...,...,...,...
1020,raging-bolt,https://pokeapi.co/api/v2/pokemon-species/1021/,9,1021
1021,iron-boulder,https://pokeapi.co/api/v2/pokemon-species/1022/,9,1022
1022,iron-crown,https://pokeapi.co/api/v2/pokemon-species/1023/,9,1023
1023,terapagos,https://pokeapi.co/api/v2/pokemon-species/1024/,9,1024


In [38]:
def collect_pokemon_data(pokemon_id):
    try: 
        details_url = f'https://pokeapi.co/api/v2/pokemon/{pokemon_id}'
        details_response = requests.get(details_url)
        if details_response.status_code == 200:
            details_data = details_response.json()
            details_data = {key: value for key, value in details_data.items() if 'sprites' not in key}
            with open(f'../../data/Pokemon Data/pokemon_details/pokemon_{pokemon_id}_details.json', 'w') as file:
                json.dump(details_data, file)

        species_url = f'https://pokeapi.co/api/v2/pokemon-species/{pokemon_id}'
        species_response = requests.get(species_url)
        if species_response.status_code == 200:
            species_data = species_response.json()
            english_entry = next(entry for entry in species_data['flavor_text_entries'] if entry['language']['name'] == 'en')
            species_data['flavor_text_entries'] = english_entry
            with open(f'../../data/Pokemon Data/species_details/pokemon_{pokemon_id}_species_details.json', 'w') as file:
                json.dump(species_data, file)       

        return 
    
    except Exception as e: 
        return f'Error for ID {pokemon_id}: {e}'

In [39]:
def main(pokemon_ids): 
    with ThreadPoolExecutor() as executor: 
        results = list(executor.map(collect_pokemon_data, pokemon_ids))
    return results

In [40]:
pokemon_ids = list(range(1,1026))
results = main(pokemon_ids)
print(f'Data successfully obtained for {results.count(None)} Pokemon.')

Data successfully obtained for 1025 Pokemon.
